In [1]:
# Module 4 Jupyter Lab¶

# This notebook is a place to practice concepts from Module 4. Note that you are welcome (even encouraged) to remix and reuse code from the lecture notebooks! 


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)

In [2]:
# Investigate a dataset
# We will work again with a sample from the Movie Lens dataset (you may remember this from Course 2 in the specialization). 
# Begin by reading in the following datasets, then merging them into one pandas DataFrame on the key movieID:
    # data/movielens/movies.csv
    # data/movielens/ratings.csv

movies = pd.read_csv('data/movielens/movies.csv')
ratings = pd.read_csv('data/movielens/ratings.csv')
movies_with_ratings = movies.merge(ratings, on='movieId')
movies_with_ratings

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483
...,...,...,...,...,...,...
100831,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,184,4.0,1537109082
100832,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,184,3.5,1537109545
100833,193585,Flint (2017),Drama,184,3.5,1537109805
100834,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,184,3.5,1537110021


In [3]:
two_movie_ratings = movies_with_ratings[movies_with_ratings['title'].isin(["Forrest Gump (1994)","Shawshank Redemption, The (1994)"])]
two_movie_ratings

,movieId,title,genres,userId,rating,timestamp
8652,318,"Shawshank Redemption, The (1994)",Crime|Drama,2,3.0,1445714835
8653,318,"Shawshank Redemption, The (1994)",Crime|Drama,5,3.0,847434880
8654,318,"Shawshank Redemption, The (1994)",Crime|Drama,6,5.0,845553200
8655,318,"Shawshank Redemption, The (1994)",Crime|Drama,8,5.0,839463489
8656,318,"Shawshank Redemption, The (1994)",Crime|Drama,11,4.0,902155070
...,...,...,...,...,...,...
10343,356,Forrest Gump (1994),Comedy|Drama|Romance|War,605,3.0,1277097509
10344,356,Forrest Gump (1994),Comedy|Drama|Romance|War,606,4.0,1171231370
10345,356,Forrest Gump (1994),Comedy|Drama|Romance|War,608,3.0,1117162603
10346,356,Forrest Gump (1994),Comedy|Drama|Romance|War,609,4.0,847220869


In [4]:
# What is the average rating for each of these two movies? Calculate:
    # mu_f, the average rating for Forrest Gump,
    # mu_s, the average rating for the Shawshank Redemption, and
    # obs_diff, which is the difference in ratings, calculated as mu_f - mu_s

mu_s = two_movie_ratings[two_movie_ratings['movieId'] == 318].rating.mean() # Shawshank
mu_f = two_movie_ratings[two_movie_ratings['movieId'] == 356].rating.mean() # Forest Gump
obs_diff = mu_f - mu_s

mu_s, mu_f, obs_diff

(np.float64(4.429022082018927),
 np.float64(4.164133738601824),
 np.float64(-0.26488834341710366))

We do see a difference in ratings - but is it statistically significant? Let's decide using the bootstrap sampling method covered in the lectures.


# Define your hypotheses

Create your null and alternative hypotheses about whether a difference in ratings exists for the two movies.


### Null Hypothesis(Ho) : There is no real difference between the average ratings of "Forrest Gump" (mu_f) and "Shawshank Redemption" (mu_s). Any observed difference is due to random sampling chance.
    Ho : mu_f - mu_s = 0

### Alternative Hypothesis (Ha) : There is a statistically significant difference between the average ratings of the two movies.
    Ha : mu_f - mu_s <> 0


## Get ready to bootstrap

How can you use bootstrapping to simulate the null distribution corresponding to your null hypothesis? You can reuse and remix code from the lecture notebooks to help you here.

First, write a function calculate_test_statistic() that calculates the difference in ratings for the two movies (you can assume it will always operate on a DataFrame with the same features as two_movie_ratings). This function should return a numeric datatype.

In [5]:

def calculate_test_statistic(df):
    """
    Calculates the difference in mean ratings for the two movies: 
    mu_f (Forrest Gump) - mu_s (Shawshank)
    """
    mu_s = df[df['movieId'] == 318]['rating'].mean() # Shawshank Redemption
    mu_f = df[df['movieId'] == 356]['rating'].mean() # Forrest Gump
    
    return mu_f - mu_s



# Calculate the actual observed difference
obs_diff = calculate_test_statistic(two_movie_ratings)
print(f"Observed Difference: {obs_diff:.4f}") 



Observed Difference: -0.2649


Next, write a function simulate_shuffling() that randomly shuffles (meaning, sample them without replacement) the ratings associated with each row in a DataFrame. You can continue to assume this function will operate on DataFrames with the same features as two_movie_ratings. This function should return a shuffled DataFrame. 

In [6]:

def simulate_shuffling(df):
    """
    Randomly shuffles the ratings in the 'rating' column of the DataFrame.
    This simulates the null hypothesis where movie identity has no 
    impact on the rating.
    """
    # Create a copy to ensure the original DataFrame remains unchanged
    shuffled_df = df.copy()
    
    # Shuffle the 'rating' column (sampling without replacement)
    shuffled_df['rating'] = rng.permutation(df['rating'].values)
    
    return shuffled_df

### Simulate the null distribution

Using the code you've written below, now simulate 1000 boostrap samples via shuffling and calculate their test statistics. Save the resulting test statistics in a variable called null_distribution.

In [7]:
# Initialize the variable to store results
null_distribution = []

# Run the simulation 1000 times
for i in range(1000):
    # 1. Shuffle the ratings using our function
    shuffled_temp_df = simulate_shuffling(two_movie_ratings)
    
    # 2. Calculate the difference in means for this shuffled data
    resampled_stat = calculate_test_statistic(shuffled_temp_df)
    
    # 3. Add to our distribution list
    null_distribution.append(resampled_stat)

# Convert to a numpy array for easier statistical analysis
null_distribution = np.array(null_distribution)

## Calculate the p-value of your observed statistic
Using the value of `obs_diff` from the original dataset and your simulated null distribution, calculate the two-sided p-value associated with `obs_diff`. 

In [8]:
# Calculate two-sided p-value
p_value = np.mean(np.abs(null_distribution) >= np.abs(obs_diff))

print(f"Observed Difference: {obs_diff:.4f}")
print(f"Two-sided P-value: {p_value}")

Observed Difference: -0.2649
Two-sided P-value: 0.0


# Interpret your hypothesis test
Using the common threshold of $a = 0.05$, what is the result of your hypothesis test? This should be a statement about accepting or rejecting the hypotheses you created earlier.

Based on the hypothesis test performed with the bootstrapping/shuffling method, here is the result using the common threshold of α=0.05:
Result of the Hypothesis Test

    Comparison: The calculated p-value is 0.0, which is significantly less than the threshold of α=0.05.

    Decision: We reject the null hypothesis (H0​).

    Conclusion: We have sufficient statistical evidence to conclude that there is a significant difference between the average ratings of "Forrest Gump" and "The Shawshank Redemption".

In plain terms, the difference of −0.265 in their mean ratings is not due to random chance. The data suggests that "The Shawshank Redemption" is rated consistently higher by users in a way that is statistically meaningful.